# Learning Rate V.S. Temporal Difference Learning Steps Analysis

This notebook analyzes the relationship between learning rate and temporal difference learning steps.

## Objective

- Understand how different learning rates affect the number of training steps required for agent convergence in the ARC environment.
- Identify optimal learning rate in combination with TD learning steps for stable training.

## Methodology

### Dataset

Using the standard ARC dataset V1, we randomly select 100 training examples.

### Hyperparameter Grid

We will keep the default hyperparameters constant while varying only:

- Learning rate (e.g., 1e-3, 3e-4, 1e-4, 3e-5)
- Temporal difference learning steps (e.g., 1, 3, 5, 10)
- Training episodes = 10000
- Epochs = 10
- Batch size = 32

### Metrics to Track

- Training loss curves
- Reward progression
- Convergence speed
- Training stability (variance in loss/reward)
- Success rate (percentage of episodes where agent solves the task)


# Sample path list

In [ ]:
import os


def load_arc_sample(arc_agi_1_path, arc_agi_2_path):
    """Loads a sample of ARC files from the ARC AGI datasets."""
    with open("./arc_sample.txt", "r") as f:
        return f.read().split("\n")


def create_arc_sample(arc_agi_1_path, arc_agi_2_path, sample_size=50):
    """Creates a sample of ARC files from the ARC AGI datasets."""
    import random

    random.seed(42)
    arc_sample = [
        os.path.join(arc_agi_1_path, arc_file)
        for arc_file in random.sample(os.listdir(arc_agi_1_path), sample_size)
    ]
    arc_sample += [
        os.path.join(arc_agi_2_path, arc_file)
        for arc_file in random.sample(os.listdir(arc_agi_2_path), sample_size)
    ]
    with open("./arc_sample.txt", "w") as f:
        f.write("\n".join(arc_sample))
    return arc_sample


ARC_AGI_1_DATASET_PATH = os.path.join(
    os.environ.get("ARC_AGI_1_DATASET_PATH"), "training"
)
ARC_AGI_2_DATASET_PATH = os.path.join(
    os.environ.get("ARC_AGI_2_DATASET_PATH"), "training"
)
try:
    arc_sample = load_arc_sample(ARC_AGI_1_DATASET_PATH, ARC_AGI_2_DATASET_PATH)
except FileNotFoundError:
    arc_sample = create_arc_sample(ARC_AGI_1_DATASET_PATH, ARC_AGI_2_DATASET_PATH)

# Setup tarball with arc samples


In [ ]:
import os

from boto3.session import Session
from sagemaker.core.training.configs import OutputDataConfig

from aws_sagemaker import utils
from aws_sagemaker.launch import SagemakerExperiment


In [ ]:
session = Session()

aws_sagemaker_bucket = os.getenv("AWS_SAGEMAKER_BUCKET")
sagemaker_role_arn = os.getenv("AWS_SAGEMAKER_ROLE_ARN")
image_uri = "{}:latest".format(os.getenv("AWS_ECR_IMAGE_URI"))
experiment_location = "lr_experiment"

tar_s3_uri = utils.create_and_upload_arc_tar(
    session,
    "arc_sample.txt",
    aws_sagemaker_bucket,
    os.path.join(experiment_location, "arc_sample.tar.gz"),
    overwrite=False,
)
print("Dataset uploaded to: ", tar_s3_uri)

In [ ]:
experiment_name = "lr_1e3_nstep_1"
output_data_config = OutputDataConfig(
    s3_output_path=f"s3://{aws_sagemaker_bucket}/lr_experiment/{experiment_name}",
    compression_type="GZIP",
)
sagemaker_experiment = SagemakerExperiment(
    role_arn=sagemaker_role_arn,
    experiment=experiment_name,
    experiments_config="./lr_experiment/experiments.yaml",
    output_data_config=output_data_config,
    training_image=image_uri,
    epochs=10,
    instance_type="ml.g5.12xlarge",  # 4× A10G GPUs
    enable_spot_instance=True,
    instance_count=1,
    max_run=2 * 60 * 60,  # 2 hours max
    max_wait=4 * 60 * 60,  # 4 hours wait for spot
)


In [ ]:
sagemaker_experiment.train(tar_s3_uri, wait=False, logs=True)